# CRISP-DM — Risque de panne par équipement

Ce notebook est limité à la classification du risque de panne. 

## 1. Business Understanding

Prédire si le nombre de curatifs du mois suivant dépassera la médiane historique de chaque équipement, afin de prioriser la maintenance.

In [2]:
# 2. Data Understanding et 3. Data Preparation
# La fonction build_dataset de retrain_model conserve toutes les variables utiles :
# lags 1-3, fenêtres glissantes 3 mois, MTBF, disponibilité, arrêts,
# ratio curatif/préventif, volumes et durées.
from retrain_model import get_engine, build_dataset

engine = get_engine()
interventions, training, scoring, features = build_dataset(engine)
print(f'Equipements : {interventions.equip_id.nunique()}')
print(f'Observations : {len(training)} | Features : {len(features)}')
training[['equip_id', 'annee_mois', 'curatif_mois_suivant']].head()

Equipements : 938
Observations : 2998 | Features : 21


,equip_id,annee_mois,curatif_mois_suivant
1,9,2026-02-01,0
1,12,2025-02-01,0
2,12,2025-03-01,0
3,12,2025-04-01,0
4,12,2025-05-01,0


In [3]:
# 4. Modeling et 5. Evaluation
# Split temporel 80/20 ; comparaison des modèles ; production = Random Forest 
training = training.sort_values(['annee_mois', 'equip_id']).reset_index(drop=True)
split = int(len(training) * 0.8)
print(f'Train temporel : {split} | Test temporel : {len(training) - split}')
print('Cible : curatifs M+1 > médiane historique de l équipement')

Train temporel : 2398 | Test temporel : 600
Cible : curatifs M+1 > médiane historique de l équipement


In [4]:
# 6. Deployment
# Lance le pipeline complet : champion/challenger (tolérance AUC 0.005),
# prédictions calibrées et catégories par percentiles 66 et 85.
from retrain_model import main
main()

# Sorties : best_model.pkl, predictions_risque.csv et model_meta.json.

RÉENTRAÎNEMENT — RISQUE DE PANNE PAR ÉQUIPEMENT
CRISP-DM : Business Understanding > Data Understanding > Data Preparation
           Modeling > Evaluation > Deployment
Cible : curatifs du mois suivant > médiane historique de l'équipement

[1/4] Extraction et préparation des données...
  Équipements : 938 | interventions : 8,781
  Observations entraînement : 2,998 | features : 21
  Classes : risque cible=1 : 706 (23.5 %) | cible=0 : 2292

[2/4] Modeling — split temporel : train=2,398, test=600
  Benchmark des modèles en cours...
    Logistic Regression: AUC=0.6469 | F1=0.5521 | Precision=0.3893 | Recall=0.9489
    Random Forest: AUC=0.6936 | F1=0.5754 | Precision=0.4140 | Recall=0.9432
    Gradient Boosting + Calibration: AUC=0.6612 | F1=0.5322 | Precision=0.3792 | Recall=0.8920

[3/4] Evaluation — Meilleur modèle: Random Forest (AUC train=0.8646 | AUC test=0.6936)
  Champion/challenger : AUC champion=0.6936 | challenger accepté (tolérance 0.005)

[4/4] Deployment — génération des prédi